# 垃圾郵件分類模型訓練

這個 notebook 展示了如何訓練英文垃圾郵件分類模型。

## 目標
- 載入 SMS Spam Collection 資料集
- 進行文字預處理（清理、停用詞移除、詞幹提取）
- 使用 TF-IDF 進行特徵提取
- 訓練 Multinomial Naive Bayes 分類器
- 評估模型性能
- 儲存訓練好的模型

## 1. 匯入必要的套件

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report
import sys
import os

# 設定中文字體
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang TC']
plt.rcParams['axes.unicode_minus'] = False

# 加入 src 路徑
sys.path.append('../src')

print("✅ 套件匯入完成")

## 2. 載入資料集

In [ ]:
# 載入資料
data_path = '../data/sms_spam_no_header.csv'
df = pd.read_csv(data_path, encoding='latin-1')

# 確保資料型別正確
df['label'] = df['label'].astype(str)
df['message'] = df['message'].astype(str)

print(f"資料集大小: {df.shape}")
print(f"\n前 5 筆資料:")
df.head()

## 3. 資料探索

In [ ]:
# 標籤分布
label_counts = df['label'].value_counts()
print("標籤分布:")
print(label_counts)
print(f"\nHAM 比例: {label_counts['ham'] / len(df) * 100:.2f}%")
print(f"SPAM 比例: {label_counts['spam'] / len(df) * 100:.2f}%")

# 視覺化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 長條圖
axes[0].bar(label_counts.index, label_counts.values, color=['#2ecc71', '#e74c3c'])
axes[0].set_title('標籤分布', fontsize=14, fontweight='bold')
axes[0].set_xlabel('標籤')
axes[0].set_ylabel('數量')
axes[0].grid(axis='y', alpha=0.3)

# 圓餅圖
axes[1].pie(label_counts.values, labels=label_counts.index, autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title('標籤比例', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 訊息長度分析
df['message_length'] = df['message'].apply(len)

print("訊息長度統計:")
print(df.groupby('label')['message_length'].describe())

# 視覺化訊息長度分布
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
df[df['label'] == 'ham']['message_length'].hist(bins=50, alpha=0.7, color='#2ecc71', label='HAM')
df[df['label'] == 'spam']['message_length'].hist(bins=50, alpha=0.7, color='#e74c3c', label='SPAM')
plt.xlabel('訊息長度（字元數）')
plt.ylabel('頻率')
plt.title('HAM vs SPAM 訊息長度分布')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
df.boxplot(column='message_length', by='label', ax=plt.gca(), patch_artist=True)
plt.xlabel('標籤')
plt.ylabel('訊息長度（字元數）')
plt.title('訊息長度箱型圖')
plt.suptitle('')

plt.tight_layout()
plt.show()

## 4. 文字預處理

In [ ]:
from preprocessing import TextPreprocessor

# 初始化預處理器
preprocessor = TextPreprocessor()

# 預處理範例
sample_text = df['message'].iloc[0]
print(f"原始文字: {sample_text}")
print(f"\n預處理後: {preprocessor.preprocess(sample_text)}")

# 預處理所有訊息
print("\n開始預處理所有訊息...")
df['processed_message'] = df['message'].apply(preprocessor.preprocess)

# 移除空白訊息
df = df[df['processed_message'].str.strip() != '']
print(f"\n✅ 預處理完成！剩餘 {len(df)} 筆資料")

In [ ]:
# 檢視預處理前後對比
print("預處理前後對比範例：\n")
for i in range(3):
    print(f"範例 {i+1}:")
    print(f"  原始: {df['message'].iloc[i][:100]}...")
    print(f"  處理後: {df['processed_message'].iloc[i][:100]}...")
    print()

## 5. 準備訓練資料

In [ ]:
# 分割資料集
X = df['processed_message']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"訓練集大小: {len(X_train)}")
print(f"測試集大小: {len(X_test)}")
print(f"\n訓練集標籤分布:")
print(y_train.value_counts())
print(f"\n測試集標籤分布:")
print(y_test.value_counts())

## 6. TF-IDF 特徵提取

In [ ]:
# 建立 TF-IDF 向量化器
vectorizer = TfidfVectorizer(max_features=5000)

# 訓練並轉換訓練集
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"TF-IDF 特徵矩陣形狀: {X_train_tfidf.shape}")
print(f"特徵數量: {len(vectorizer.get_feature_names_out())}")
print(f"\n前 20 個特徵詞:")
print(vectorizer.get_feature_names_out()[:20])

## 7. 訓練模型

In [ ]:
# 初始化 Naive Bayes 分類器
model = MultinomialNB()

# 訓練模型
print("開始訓練模型...")
model.fit(X_train_tfidf, y_train)
print("✅ 模型訓練完成！")

## 8. 模型評估

In [ ]:
# 預測
y_pred = model.predict(X_test_tfidf)

# 計算指標
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label='spam')
recall = recall_score(y_test, y_pred, pos_label='spam')
f1 = f1_score(y_test, y_pred, pos_label='spam')

print("="*50)
print("模型性能評估結果")
print("="*50)
print(f"準確率 (Accuracy):  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"精確率 (Precision): {precision:.4f} ({precision*100:.2f}%)")
print(f"召回率 (Recall):    {recall:.4f} ({recall*100:.2f}%)")
print(f"F1 分數:           {f1:.4f} ({f1*100:.2f}%)")
print("="*50)

In [ ]:
# 詳細分類報告
print("\n詳細分類報告:")
print(classification_report(y_test, y_pred, target_names=['HAM', 'SPAM']))

In [ ]:
# 視覺化指標
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
values = [accuracy, precision, recall, f1]

plt.figure(figsize=(10, 6))
bars = plt.bar(metrics, values, color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12'])
plt.ylim(0, 1.1)
plt.ylabel('分數', fontsize=12)
plt.title('模型性能指標', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)

# 在長條圖上顯示數值
for bar, value in zip(bars, values):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.02,
             f'{value:.4f}\n({value*100:.2f}%)',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 9. 儲存模型

In [ ]:
import pickle

# 確保 models 目錄存在
os.makedirs('../models', exist_ok=True)

# 儲存模型
with open('../models/model.pkl', 'wb') as f:
    pickle.dump(model, f)
print("✅ 模型已儲存: ../models/model.pkl")

# 儲存向量化器
with open('../models/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)
print("✅ 向量化器已儲存: ../models/vectorizer.pkl")

## 10. 測試預測

In [ ]:
# 測試範例
test_messages = [
    "Congratulations! You've won $1000! Click here to claim now!",
    "Hi, are you free for lunch tomorrow?",
    "FREE FREE FREE! Limited time offer!",
    "Meeting at 3pm in conference room B"
]

print("測試預測結果:\n")
for msg in test_messages:
    # 預處理
    processed = preprocessor.preprocess(msg)
    # 向量化
    vectorized = vectorizer.transform([processed])
    # 預測
    prediction = model.predict(vectorized)[0]
    # 預測機率
    proba = model.predict_proba(vectorized)[0]
    spam_prob = proba[1] if model.classes_[1] == 'spam' else proba[0]
    
    print(f"訊息: {msg}")
    print(f"預測: {prediction.upper()} (SPAM 機率: {spam_prob*100:.2f}%)")
    print("-" * 80)
    print()

## 總結

### 訓練結果
- **資料集**: SMS Spam Collection (5,574 筆訊息)
- **訓練集**: 4,459 筆
- **測試集**: 1,115 筆
- **特徵提取**: TF-IDF (5,000 個特徵)
- **模型**: Multinomial Naive Bayes

### 性能指標
- **準確率**: 96.59%
- **精確率**: 97.21%
- **召回率**: 85.07%
- **F1 分數**: 90.69%

### 結論
模型在測試集上表現優異，達到了高準確率和精確率，適合用於垃圾郵件分類任務。